In [0]:
# ===========================================
# Notebook Name:
# 04_build_archetype_period_summary
#
# Purpose:
# Aggregate archetype (deck cluster identity)
# share of tournament results over time, at
# both weekly and monthly grain, so metagame
# trends can be tracked without re-running the
# ad-hoc query in 99_analysis every time.
#
# Only the latest clustering run
# (model_run_id) is summarized. Archetype
# identity is resolved through
# gold.v_deck_archetypes_named so a cluster's
# human-reviewed name survives even though its
# cluster_id is unstable across reruns.
#
# Input:
# pokemon.gold.v_deck_archetypes_named
# pokemon.silver.tournament_results
# pokemon.silver.tournaments
# pokemon.silver.decks
#
# Output:
# pokemon.gold.archetype_period_summary
#
# Grain:
# One row per period_type x period_start_date x archetype_label
# ===========================================

In [0]:
from pyspark.sql import functions as F

DECK_ARCHETYPES_NAMED_VIEW = (
    "pokemon.gold.v_deck_archetypes_named"
)
TOURNAMENT_RESULTS_TABLE = (
    "pokemon.silver.tournament_results"
)
TOURNAMENTS_TABLE = "pokemon.silver.tournaments"
DECKS_TABLE = "pokemon.silver.decks"

ARCHETYPE_PERIOD_SUMMARY_TABLE = (
    "pokemon.gold.archetype_period_summary"
)

print("Input :", DECK_ARCHETYPES_NAMED_VIEW)
print("Input :", TOURNAMENT_RESULTS_TABLE)
print("Input :", TOURNAMENTS_TABLE)
print("Input :", DECKS_TABLE)
print("Output:", ARCHETYPE_PERIOD_SUMMARY_TABLE)

In [0]:
deck_archetypes_named_df = spark.table(
    DECK_ARCHETYPES_NAMED_VIEW
)
tournament_results_df = spark.table(
    TOURNAMENT_RESULTS_TABLE
)
tournaments_df = spark.table(TOURNAMENTS_TABLE)
decks_df = spark.table(DECKS_TABLE)

print(
    "Deck archetype rows:",
    deck_archetypes_named_df.count(),
)
print(
    "Tournament result rows:",
    tournament_results_df.count(),
)
print("Tournament rows:", tournaments_df.count())
print("Deck rows:", decks_df.count())

In [0]:
latest_model_run_row = (
    deck_archetypes_named_df
    .select("model_run_id", "clustered_at")
    .distinct()
    .orderBy(F.col("clustered_at").desc())
    .first()
)

if latest_model_run_row is None:
    raise ValueError(
        f"{DECK_ARCHETYPES_NAMED_VIEW} is empty. "
        "Run 04_ml/00_build_deck_archetypes first."
    )

latest_model_run_id = latest_model_run_row[
    "model_run_id"
]

latest_archetypes_df = (
    deck_archetypes_named_df
    .filter(
        F.col("model_run_id") == latest_model_run_id
    )
    .withColumn(
        "archetype_label",
        F.coalesce(
            F.col("archetype_name"),
            F.lit("(pending review)"),
        ),
    )
)

print("Summarizing model_run_id:", latest_model_run_id)

In [0]:
# event_date_week is a raw, unvalidated
# passthrough string from the source API
# (see 02_silver/00_build_tournaments) and its
# format/reliability is not guaranteed. Derive
# period boundaries from event_date instead,
# matching the effective_event_date pattern
# already used in 03_gold/01_build_deck_registry.
tournament_dates_df = (
    tournaments_df
    .select(
        "tournament_id",
        "event_date",
        "source_scraped_at",
    )
    .withColumn(
        "effective_event_date",
        F.coalesce(
            F.col("event_date"),
            F.to_date("source_scraped_at"),
        ),
    )
)

# tournament_results.deck_hash is not populated
# in production (a pre-existing, unrelated data
# issue that also affects gold.deck_registry's
# tournament-performance columns). deck_code IS
# populated, so deck_hash is resolved by joining
# through silver.decks instead of trusting the
# tournament_results.deck_hash column directly.
tournament_result_decks_df = (
    tournament_results_df
    .select("tournament_id", "deck_code")
    .join(
        decks_df.select("deck_code", "deck_hash"),
        on="deck_code",
        how="inner",
    )
)

deck_result_archetypes_df = (
    tournament_result_decks_df
    .select("tournament_id", "deck_hash")
    .join(
        tournament_dates_df.select(
            "tournament_id",
            "effective_event_date",
        ),
        on="tournament_id",
        how="inner",
    )
    .join(
        latest_archetypes_df.select(
            "deck_hash",
            "archetype_label",
        ),
        on="deck_hash",
        how="inner",
    )
)

display(
    deck_result_archetypes_df.orderBy(
        "effective_event_date"
    )
)

In [0]:
# Weekly and monthly grain share the same
# aggregation logic and only differ in how the
# period boundary is truncated, so both grains
# are built from one function instead of two
# near-duplicate blocks.
def build_period_summary(period_type, trunc_unit):
    period_df = (
        deck_result_archetypes_df
        .withColumn(
            "period_start_date",
            F.trunc(
                F.col("effective_event_date"),
                trunc_unit,
            ),
        )
        .groupBy(
            "period_start_date",
            "archetype_label",
        )
        .agg(
            F.countDistinct("deck_hash").alias(
                "deck_count"
            ),
            F.count("*").alias("result_count"),
        )
    )

    period_total_df = (
        period_df
        .groupBy("period_start_date")
        .agg(
            F.sum("result_count").alias(
                "period_total_result_count"
            )
        )
    )

    period_end_expr = (
        F.last_day("period_start_date")
        if trunc_unit == "month"
        else F.date_add("period_start_date", 6)
    )

    return (
        period_df
        .join(
            period_total_df,
            on="period_start_date",
            how="left",
        )
        .withColumn(
            "share_pct",
            F.round(
                F.col("result_count") * 100.0
                / F.col("period_total_result_count"),
                1,
            ),
        )
        .withColumn(
            "period_end_date", period_end_expr
        )
        .withColumn(
            "period_type", F.lit(period_type)
        )
    )


weekly_summary_df = build_period_summary(
    "week", "week"
)
monthly_summary_df = build_period_summary(
    "month", "month"
)

archetype_period_summary_df = (
    weekly_summary_df
    .unionByName(monthly_summary_df)
    .withColumn(
        "model_run_id",
        F.lit(latest_model_run_id),
    )
    .withColumn(
        "updated_at", F.current_timestamp()
    )
    .select(
        "period_type",
        "period_start_date",
        "period_end_date",
        "archetype_label",
        "deck_count",
        "result_count",
        "period_total_result_count",
        "share_pct",
        "model_run_id",
        "updated_at",
    )
)

display(
    archetype_period_summary_df.orderBy(
        "period_type",
        "period_start_date",
        F.col("share_pct").desc(),
    )
)

In [0]:
# The checks below (duplicate rows, share_pct
# summing to ~100, null keys) are all vacuously
# true on an empty DataFrame, since there are no
# rows to violate any of them. A dedicated
# non-emptiness check is required so a broken
# upstream join (e.g. deck_hash resolution)
# fails loudly instead of silently producing an
# empty table that still reports success.
archetype_period_summary_row_count = (
    archetype_period_summary_df.count()
)

if archetype_period_summary_row_count == 0:
    raise ValueError(
        "archetype_period_summary_df is empty. "
        "This likely means the join between "
        f"{TOURNAMENT_RESULTS_TABLE} and "
        f"{DECK_ARCHETYPES_NAMED_VIEW} produced "
        "no matches -- check that deck_hash is "
        "populated on both sides."
    )

print(
    "Validation passed: "
    f"{archetype_period_summary_row_count} rows "
    "produced"
)

In [0]:
duplicate_period_rows_df = (
    archetype_period_summary_df
    .groupBy(
        "period_type",
        "period_start_date",
        "archetype_label",
    )
    .count()
    .filter(F.col("count") > 1)
)

duplicate_period_row_count = (
    duplicate_period_rows_df.count()
)

if duplicate_period_row_count > 0:
    display(duplicate_period_rows_df)

    raise ValueError(
        f"{duplicate_period_row_count} duplicated "
        "(period_type, period_start_date, "
        "archetype_label) rows detected"
    )

print(
    "Validation passed: one row per "
    "period_type x period_start_date x "
    "archetype_label"
)

In [0]:
SHARE_PCT_TOLERANCE = 1.0

period_share_totals_df = (
    archetype_period_summary_df
    .groupBy("period_type", "period_start_date")
    .agg(
        F.sum("share_pct").alias(
            "total_share_pct"
        )
    )
    .filter(
        F.abs(
            F.col("total_share_pct") - 100.0
        )
        > SHARE_PCT_TOLERANCE
    )
)

invalid_period_count = (
    period_share_totals_df.count()
)

if invalid_period_count > 0:
    display(period_share_totals_df)

    raise ValueError(
        f"{invalid_period_count} periods have "
        "share_pct not summing to ~100 "
        f"(tolerance {SHARE_PCT_TOLERANCE})"
    )

print(
    "Validation passed: share_pct sums to "
    "~100 per period"
)

In [0]:
null_key_rows_df = archetype_period_summary_df.filter(
    F.col("period_type").isNull()
    | F.col("period_start_date").isNull()
    | F.col("archetype_label").isNull()
    | F.col("model_run_id").isNull()
)

null_key_row_count = null_key_rows_df.count()

if null_key_row_count > 0:
    display(null_key_rows_df)

    raise ValueError(
        f"{null_key_row_count} rows have a "
        "null key column"
    )

print("Validation passed: no null key columns")

In [0]:
# -------------------------------------------
# Rebuild archetype_period_summary atomically
# via CREATE OR REPLACE TABLE AS SELECT,
# instead of TRUNCATE + append. If the write
# fails partway, the old table contents are
# left intact rather than a table that has
# been emptied but not refilled.
# -------------------------------------------
archetype_period_summary_df.createOrReplaceTempView(
    "archetype_period_summary_staging"
)

spark.sql(f"""
CREATE OR REPLACE TABLE {ARCHETYPE_PERIOD_SUMMARY_TABLE}
COMMENT 'Archetype share of tournament results by week and month, for the latest clustering run'
AS SELECT * FROM archetype_period_summary_staging
""")

spark.sql(
    f"ALTER TABLE {ARCHETYPE_PERIOD_SUMMARY_TABLE} "
    "ALTER COLUMN period_type SET NOT NULL"
)
spark.sql(
    f"ALTER TABLE {ARCHETYPE_PERIOD_SUMMARY_TABLE} "
    "ALTER COLUMN period_start_date SET NOT NULL"
)
spark.sql(
    f"ALTER TABLE {ARCHETYPE_PERIOD_SUMMARY_TABLE} "
    "ALTER COLUMN archetype_label SET NOT NULL"
)
spark.sql(
    f"ALTER TABLE {ARCHETYPE_PERIOD_SUMMARY_TABLE} "
    "ALTER COLUMN model_run_id SET NOT NULL"
)

print(
    "Gold archetype_period_summary table "
    "rebuilt atomically."
)

In [0]:
display(
    spark.table(ARCHETYPE_PERIOD_SUMMARY_TABLE)
    .orderBy(
        "period_type",
        "period_start_date",
        F.col("share_pct").desc(),
    )
)